# IxIDN benchmarks: composition, ROC curves and per-disease lookups

This notebook explores the trial co-investigation benchmarks used in the paper:

* **IxIDN+**: disease pairs trialled with the same drug.
* **IxIDN−**: candidate non-pairs, ≥ 3 links apart in the trial graph with no shared-mechanism drug.
* **IxIDN-Matched**: the 4,364 blinded pairs (2,182 + 2,182) matched on degree and other covariates; this is the evaluation set for main Figures 3, 5 and 6.

It reads only repository paths (`data/derived`, `data/zenodo`). Numerical parity with the manuscript is checked in `tests/`, not here.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

ROOT = Path.cwd().resolve()
while not (ROOT / "data").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DV, ZF = ROOT / "data" / "derived", ROOT / "data" / "zenodo" / "figures_main"
F3, F5, F6 = DV / "fig3_ixidn", DV / "fig5_clustering", DV / "fig6_comparison"
print(ROOT)

## 1. Benchmark construction funnel

Pair counts at each stage, from the trial graph (stage 1) to the degree-matched tune and test splits (stage 5).

In [ ]:
funnel = pd.read_csv(F3 / "funnel_counts.tsv", sep="\t")
funnel[["stage", "class", "pairs"]]

## 2. Composition of IxIDN+, IxIDN− and IxIDN-Matched

For each set: number of pairs, distinct diseases, ontology source of the disease identifiers (MONDO, OMIM, Orphanet/ORDO), and the share of rare diseases (from the disease map annotation).

In [ ]:
dmap = pd.read_csv(F5 / "disease_map_k55.tsv.gz", sep="\t").set_index("mondo_id")
pos1 = pd.read_csv(F3 / "stage1_ixidn_pos.tsv.gz", sep="\t").rename(columns={"disease_a": "a", "disease_b": "b"})
pos3 = pd.read_csv(F3 / "stage3_ixidn_pos.tsv.gz", sep="\t")
neg3 = pd.read_csv(ZF / "stage3_ixidn_neg.tsv.gz", sep="\t")
matched = pd.read_csv(F3 / "ixidn_matched_pairs.tsv", sep="\t", dtype={"cellA": str})
sets = {"IxIDN+ (trial graph)": pos1, "IxIDN+ (candidate universe)": pos3, "IxIDN- (candidate universe)": neg3,
        "IxIDN-Matched +": matched[matched.label == 1], "IxIDN-Matched -": matched[matched.label == 0]}

def composition(df):
    d = pd.Series(pd.unique(df[["a", "b"]].values.ravel()))
    src = d.str.split(":").str[0].value_counts(normalize=True)
    rare = dmap.reindex(d).is_rare
    return {"pairs": len(df), "diseases": len(d), **{f"% {k}": round(100 * v, 1) for k, v in src.items()},
            "% rare (annotated)": round(100 * rare.mean(), 1), "% unannotated": round(100 * rare.isna().mean(), 1)}

pd.DataFrame({k: composition(v) for k, v in sets.items()}).T.fillna(0)

Within IxIDN-Matched, pairs are labelled by the rarity of their two diseases (rare–rare, rare–common, common–common), the strata of Figure 5c–d.

In [ ]:
sc = pd.read_csv(F5 / "ixidn_matched_pair_scores.tsv", sep="\t")
pd.crosstab(sc.rare_common_pair, sc.y.map({1: "IxIDN+", 0: "IxIDN-"}), margins=True)

In [ ]:
pd.crosstab(sc.pool, sc.y.map({1: "IxIDN+", 0: "IxIDN-"}), margins=True)

## 3. ROC curves behind Figure 3: why matching is needed

Before matching, disease degree alone separates IxIDN+ from IxIDN− almost perfectly on the stage-3 candidates. After matching, single properties are close to chance (Figure 3d) while DisAtlas still separates the pairs.

In [ ]:
cand = pd.read_parquet(ZF / "stage3_candidates_degree.parquet")
fig, ax = plt.subplots(figsize=(4.2, 4.2))
for lab, y, s in [("degree sum, stage-3 candidates", cand.label, cand.deg_a + cand.deg_b),
                  ("DisAtlas, IxIDN-Matched", sc.y, sc.p),
                  ("weighted Adamic-Adar, IxIDN-Matched", sc.y, sc.waa)]:
    f, t, _ = roc_curve(y, s); ax.plot(f, t, label=f"{lab} ({roc_auc_score(y, s):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.legend(fontsize=7, loc="lower right"); fig.tight_layout()

In [ ]:
pd.read_csv(F3 / "shortcut_diagnostics.csv")

## 4. ROC curves behind Figure 6: DisAtlas and comparator systems

`per_run_scores.tsv` holds one column per system and run (`system|run`). As in Figure 6a, multi-run systems are averaged over runs and pairs a system did not score are ranked last. The node-disjoint DisAtlas arm (benchmark diseases held out of training) drops the 7 pairs it cannot score.

In [ ]:
R = pd.read_csv(F6 / "per_run_scores.tsv", sep="\t")
runs = [c for c in R.columns if "|" in c]
systems = sorted({c.split("|")[0] for c in runs})
def system_score(s):
    cols = [c for c in runs if c.split("|")[0] == s]
    m = R[cols].mean(axis=1)
    return m.fillna(m.min() - 1)            # unscored pairs ranked last
ND = pd.read_csv(F6 / "node_disjoint_per_seed.tsv", sep="\t").merge(R[["blind_id"]], on="blind_id")
nd_score = ND[[c for c in ND.columns if "|" in c]].mean(axis=1)

fig, ax = plt.subplots(figsize=(4.6, 4.6)); auc = {}
for s in systems:
    f, t, _ = roc_curve(R.y, system_score(s)); auc[s] = roc_auc_score(R.y, system_score(s)); ax.plot(f, t, lw=1.2, label=s)
ok = nd_score.notna()
f, t, _ = roc_curve(ND.y[ok], nd_score[ok]); auc["DisAtlas node-disjoint"] = roc_auc_score(ND.y[ok], nd_score[ok])
ax.plot(f, t, lw=1.2, ls=":", label="DisAtlas node-disjoint")
ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.legend(fontsize=6.5, loc="lower right"); fig.tight_layout()
pd.Series(auc, name="AUROC").sort_values(ascending=False).round(3).to_frame()

## 5. Per-disease score lookups

`pairs_for(disease)` lists every IxIDN-Matched pair involving a disease (by identifier or name substring), with its label, rarity stratum, DisAtlas score, weighted Adamic-Adar score and the score's percentile among all 4,364 pairs.

In [ ]:
names = dmap["label"].to_dict()
sc["name_a"], sc["name_b"] = sc.a.map(names), sc.b.map(names)
sc["p_percentile"] = (sc.p.rank(pct=True) * 100).round(1)

def pairs_for(q):
    hit = sc[(sc.a == q) | (sc.b == q) | sc.name_a.str.contains(q, case=False, na=False) | sc.name_b.str.contains(q, case=False, na=False)]
    return hit[["a", "name_a", "b", "name_b", "y", "rare_common_pair", "p", "p_percentile", "waa"]].sort_values("p", ascending=False)

pairs_for("myasthenia gravis").head(10)

In [ ]:
def disease_summary(q):
    h = pairs_for(q)
    return h.groupby("y").p.describe()[["count", "mean", "50%", "min", "max"]].rename(index={1: "IxIDN+", 0: "IxIDN-"})

disease_summary("Crohn")